### imports + functions

In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path
from typing import Callable

from src.context_scaling.eval_index import load_eval_index, load_eval_csvs

ModuleNotFoundError: No module named 'src'

In [ ]:
def _smooth_1d(x, window):
    if window <= 1:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")


def mean_drop_top_frac(df, frac):
    if frac <= 0:
        return df.mean(axis=0)
    q = 1.0 - frac
    return df.where(df.le(df.quantile(q, axis=0)), np.nan).mean(axis=0)


def diff(a, b):
    return a - b, "\u0394 loss"


def norm_diff(a, b):
    return (a - b) * 100 / (a + b), "normalized \u0394 loss %"


def ratio(a, b):
    return a * 100 / b, "ratio %"


def plot_diffs(
    labels: list[str],
    dfs: list[pd.DataFrame],
    compare_fn: Callable,
    smoothing: int = 1,
    head_trim: int = 0,
    tail_trim: int = 0,
    seq_len: int = 2048,
    figsize: tuple[int, int] = (12, 7),
    merge_rules: list = None,
    title: str = 'plot',
):
    assert len(labels) == len(dfs), "Labels and dfs must match in length."

    labels, dfs = merger(labels, dfs, merge_rules)

    labels, dfs = zip(*sorted(zip(labels, dfs)))
    labels = list(labels)
    dfs = list(dfs)

    avgs = []
    seq_len_adj = seq_len - smoothing

    for df in dfs:
        avg = df.mean(axis=0).values
        end = -tail_trim if tail_trim else None
        avg = avg[head_trim:end]
        avg = _smooth_1d(avg, smoothing)
        avgs.append(avg)

    lengths = [len(avg) for avg in avgs]
    print(f"lengths: {lengths}")
    print(f"adj: {seq_len_adj}")
    if not lengths or max(lengths) == 0:
        raise ValueError("No data left after trimming/smoothing.")

    long_enough = [avg[:seq_len] for avg in avgs if len(avg) >= seq_len_adj]
    if not long_enough:
        raise ValueError(f"No dfs are long enough for seq_len={seq_len} after smoothing.")

    grand_avg = np.mean(long_enough, axis=0)

    plt.figure(figsize=figsize)

    ylabel = None
    for avg, label in zip(avgs, labels):
        n = min(len(avg), seq_len)
        if n == 0:
            continue
        diff, this_ylabel = compare_fn(avg[:n], grand_avg[:n])
        if ylabel is None:
            ylabel = this_ylabel
        token_ids = np.arange(n) + head_trim
        plt.plot(token_ids, diff, label=label)

    plt.xlabel("Token ID")
    plt.ylabel(ylabel or "Difference")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def merger(labels: list[str], dfs: list[pd.DataFrame], merge_rules: list):
    if not merge_rules:
        return labels, dfs

    current_labels = labels
    current_dfs = dfs

    for key, val in merge_rules:
        next_labels = []
        next_dfs = []

        if val is not None:
            search_str = f"{key}={val}"
            for lbl, df in zip(current_labels, current_dfs):
                if search_str in lbl:
                    next_labels.append(lbl)
                    next_dfs.append(df)
        else:
            groups = {}
            for lbl, df in zip(current_labels, current_dfs):
                base_lbl = re.sub(rf"{key}=[^+._\s]+", f"{key}=avg", lbl)
                if base_lbl not in groups:
                    groups[base_lbl] = []
                groups[base_lbl].append(df)

            for base_lbl, df_group in groups.items():
                next_labels.append(base_lbl)
                merged_df = sum(df_group) / len(df_group)
                next_dfs.append(merged_df)

        current_labels = next_labels
        current_dfs = next_dfs

    return current_labels, current_dfs

## 2503_grid (index-based)

In [ ]:
eval_dir = "/home/janek/Documents/IDEAS/2503_grid_namesfix"
idx = load_eval_index(eval_dir)
print(f"index shape: {idx.shape}")
print(f"columns: {[c for c in idx.columns if c.startswith('common.')]}")
idx[["csv", "run_id", "model_step", "eval_seq_len", "common.dmodel", "common.dff", "common.kv_heads", "common.sequence_length", "trainer.learning_rate"]].head(10)

### MHA vs MQA

In [ ]:
subset = idx[idx["trainer.learning_rate"] == 12]
labels, dfs = load_eval_csvs(subset, eval_dir,
    label_cols=["common.kv_heads", "common.dmodel", "common.sequence_length"])

plot_diffs(
    labels=labels, dfs=dfs,
    compare_fn=diff, smoothing=100,
    merge_rules=[('common.dmodel', None), ('common.sequence_length', None)],
    title="MHA vs MQA",
)

### Learning Rate

In [ ]:
labels, dfs = load_eval_csvs(idx, eval_dir,
    label_cols=["trainer.learning_rate", "common.kv_heads", "common.dmodel"])

plot_diffs(
    labels=labels, dfs=dfs,
    compare_fn=diff, smoothing=100,
    merge_rules=[('common.dmodel', None), ('common.kv_heads', None)],
    title="Learning Rate",
)

### Sequence Length

In [ ]:
labels, dfs = load_eval_csvs(idx, eval_dir,
    label_cols=["common.sequence_length", "common.kv_heads", "common.dmodel"])

plot_diffs(
    labels=labels, dfs=dfs,
    compare_fn=diff, smoothing=100,
    seq_len=2048,
    merge_rules=[('trainer.learning_rate', None)],
    title="Sequence Length",
)

### Model Size

In [ ]:
labels, dfs = load_eval_csvs(idx, eval_dir,
    label_cols=["common.dmodel", "common.dff", "common.kv_heads", "common.sequence_length"])

plot_diffs(
    labels=labels, dfs=dfs,
    compare_fn=diff, smoothing=100,
    seq_len=2048,
    title="Model Size",
)

### DFF Grid

In [ ]:
subset = idx[idx["common.dmodel"] == 768]
labels, dfs = load_eval_csvs(subset, eval_dir,
    label_cols=["common.dff", "common.kv_heads", "common.sequence_length"])

plot_diffs(
    labels=labels, dfs=dfs,
    compare_fn=diff, smoothing=100,
    merge_rules=[('common.dmodel', None)],
    title="DFF Grid (dmodel=768)",
)